In [23]:
!pip install shap

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('data.csv')

 
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, GridSearchCV)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_curve, auc)
from sklearn.multiclass import OneVsRestClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor
import shap
 
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [25]:
df = pd.read_csv('data.csv', sep=';')

X_raw = df.drop(columns=['Target'])
y_raw = df['Target']

In [26]:

print("Loading Dataset")

le          = LabelEncoder()
y_enc       = le.fit_transform(y_raw)
CLASS_NAMES = le.classes_
print(f"  Label map : {dict(zip(le.classes_, le.transform(le.classes_)))}")
 
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_enc
)
print(f"  Train : {len(y_train)}   Test : {len(y_test)}")
 
scaler        = StandardScaler()
X_train_sc    = scaler.fit_transform(X_train)
X_test_sc     = scaler.transform(X_test)
X_train_sc_df = pd.DataFrame(X_train_sc, columns=X_raw.columns)
X_test_sc_df  = pd.DataFrame(X_test_sc,  columns=X_raw.columns)
 
def match_cols(keywords, all_cols):
    matched = []
    for kw in keywords:
        for c in all_cols:
            if kw.lower() in c.lower() and c not in matched:
                matched.append(c)
                break
    return matched
 
all_cols      = X_raw.columns.tolist()
academic_cols = match_cols(
    ['previous qualification (grade)', 'admission grade', 'age at enrollment',
     'daytime', 'curricular units'], all_cols)
socio_cols    = match_cols(
    ['scholarship', 'tuition fees', 'debtor', 'displaced',
     "mother's qualification", "father's qualification",
     "mother's occupation", "father's occupation"], all_cols)
macro_cols    = match_cols(['gdp', 'unemployment rate', 'inflation rate'], all_cols)
 
print(f"\n  Academic features      : {len(academic_cols)}")
print(f"  Socioeconomic features : {len(socio_cols)}")
print(f"  Macroeconomic features : {len(macro_cols)}")

Loading Dataset
  Label map : {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}
  Train : 3539   Test : 885

  Academic features      : 5
  Socioeconomic features : 8
  Macroeconomic features : 3


In [27]:
print(" Hyperparameter Tuning (GridSearchCV, 5-fold CV)")
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 
param_grids = {
    'KNN': {'n_neighbors': [3, 5, 7, 11, 15]},
    'SVM': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']},
    'LR' : {'C': [0.01, 0.1, 1, 10]},
    'RF' : {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
}
 
base_estimators = {
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'LR' : LogisticRegression(multi_class='multinomial', max_iter=1000,
                               random_state=RANDOM_STATE),
    'RF' : RandomForestClassifier(random_state=RANDOM_STATE),
}
 
best_params  = {}
best_models  = {}
cv_scores    = {}   # best CV F1 from GridSearchCV — used as validation evidence
 
for name, estimator in base_estimators.items():
    gs = GridSearchCV(estimator, param_grids[name],
                      cv=cv, scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train_sc, y_train)
    best_params[name] = gs.best_params_
    best_models[name] = gs.best_estimator_
    cv_scores[name]   = round(gs.best_score_, 4)
    print(f"  {name:4s}  Best params : {gs.best_params_}  "
          f"| Best CV F1 : {gs.best_score_:.4f}")
 
MODEL_NAMES = list(best_models.keys())

 Hyperparameter Tuning (GridSearchCV, 5-fold CV)
  KNN   Best params : {'n_neighbors': 5}  | Best CV F1 : 0.6069
  SVM   Best params : {'C': 10, 'gamma': 'scale'}  | Best CV F1 : 0.6749
  LR    Best params : {'C': 10}  | Best CV F1 : 0.6757
  RF    Best params : {'max_depth': 20, 'n_estimators': 200}  | Best CV F1 : 0.6917


In [28]:
print("RQ1")

def eval_model(model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    return {
        'Accuracy' : round(accuracy_score(yte, yp), 4),
        'Precision': round(precision_score(yte, yp, average='macro', zero_division=0), 4),
        'Recall'   : round(recall_score(yte, yp,    average='macro', zero_division=0), 4),
        'F1-Macro' : round(f1_score(yte, yp,        average='macro', zero_division=0), 4),
    }, yp
 
results_full = {}
preds_full   = {}
 
for name, model in best_models.items():
    met, yp = eval_model(model, X_train_sc, X_test_sc, y_train, y_test)
    results_full[name] = met
    preds_full[name]   = yp
    print(f"  {name:4s}  CV F1={cv_scores[name]}  ->  "
          f"Test: Acc={met['Accuracy']}  P={met['Precision']}  "
          f"R={met['Recall']}  F1={met['F1-Macro']}")
 
df_full   = pd.DataFrame(results_full).T
best_name = df_full['F1-Macro'].idxmax()
print(f"\n  >>> Best model : {best_name}  (Test F1 = {df_full.loc[best_name,'F1-Macro']})")
print(f"\n  Detailed report -- {best_name}:")
print(classification_report(y_test, preds_full[best_name], target_names=CLASS_NAMES))

RQ1
  KNN   CV F1=0.6069  ->  Test: Acc=0.6678  P=0.5802  R=0.5719  F1=0.573
  SVM   CV F1=0.6749  ->  Test: Acc=0.7186  P=0.65  R=0.6391  F1=0.6433
  LR    CV F1=0.6757  ->  Test: Acc=0.7718  P=0.713  R=0.6804  F1=0.6882
  RF    CV F1=0.6917  ->  Test: Acc=0.7729  P=0.7262  R=0.6847  F1=0.6959

  >>> Best model : RF  (Test F1 = 0.6959)

  Detailed report -- RF:
              precision    recall  f1-score   support

     Dropout       0.82      0.76      0.79       284
    Enrolled       0.57      0.36      0.44       159
    Graduate       0.79      0.93      0.85       442

    accuracy                           0.77       885
   macro avg       0.73      0.68      0.70       885
weighted avg       0.76      0.77      0.76       885



In [ ]:
print("\nAll columns:")
for c in X_raw.columns:
    print(f"  {c}")


All columns:
  Marital status
  Application mode
  Application order
  Course
  Daytime/evening attendance	
  Previous qualification
  Previous qualification (grade)
  Nacionality
  Mother's qualification
  Father's qualification
  Mother's occupation
  Father's occupation
  Admission grade
  Displaced
  Educational special needs
  Debtor
  Tuition fees up to date
  Gender
  Scholarship holder
  Age at enrollment
  International
  Curricular units 1st sem (credited)
  Curricular units 1st sem (enrolled)
  Curricular units 1st sem (evaluations)
  Curricular units 1st sem (approved)
  Curricular units 1st sem (grade)
  Curricular units 1st sem (without evaluations)
  Curricular units 2nd sem (credited)
  Curricular units 2nd sem (enrolled)
  Curricular units 2nd sem (evaluations)
  Curricular units 2nd sem (approved)
  Curricular units 2nd sem (grade)
  Curricular units 2nd sem (without evaluations)
  Unemployment rate
  Inflation rate
  GDP


In [29]:

print("Confusion Matrices")

 
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10))
fig2.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=13, fontweight='bold')
for ax, name in zip(axes2.flatten(), MODEL_NAMES):
    cm = confusion_matrix(y_test, preds_full[name])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}  (F1={results_full[name]["F1-Macro"]})')
    print(f"  {name}:\n{cm}\n")
plt.tight_layout()
fig2.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close(fig2)
print("  Saved: confusion_matrices.png")

Confusion Matrices
  KNN:
[[192  33  59]
 [ 47  34  78]
 [ 24  53 365]]

  SVM:
[[201  44  39]
 [ 40  56  63]
 [ 19  44 379]]

  LR:
[[218  29  37]
 [ 43  55  61]
 [ 14  18 410]]

  RF:
[[217  21  46]
 [ 37  58  64]
 [ 10  23 409]]

  Saved: confusion_matrices.png


In [42]:

print("ROC Curves")

 
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
colors_roc = ['#e63946', '#457b9d', '#2a9d8f']
 
fig3, axes3 = plt.subplots(2, 2, figsize=(13, 10))
fig3.suptitle('ROC Curves — One-vs-Rest per Class (Macro AUC)', fontsize=13, fontweight='bold')
 
for ax, name in zip(axes3.flatten(), MODEL_NAMES):
    ovr = OneVsRestClassifier(
        type(best_models[name])(**best_models[name].get_params())
    )
    ovr.fit(X_train_sc, y_train)
    y_score   = ovr.predict_proba(X_test_sc)
    macro_fpr = np.linspace(0, 1, 200)
    macro_tpr = np.zeros(200)
 
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors_roc[i], lw=1.5,
                label=f'{cls} (AUC={roc_auc_val:.2f})')
        macro_tpr += np.interp(macro_fpr, fpr, tpr)
        print(f"  {name} -- {cls} AUC: {roc_auc_val:.4f}")
 
    macro_tpr /= 3
    macro_auc  = auc(macro_fpr, macro_tpr)
    ax.plot(macro_fpr, macro_tpr, 'k--', lw=2,
            label=f'Macro avg (AUC={macro_auc:.2f})')
    ax.plot([0, 1], [0, 1], 'gray', lw=1, linestyle=':')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC -- {name}'); ax.legend(fontsize=8, loc='lower right')
    print(f"  {name} -- Macro AUC: {macro_auc:.4f}\n")
 
plt.tight_layout()
fig3.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig3)
print("  Saved:roc_curves.png")

ROC Curves
  KNN -- Dropout AUC: 0.8453
  KNN -- Enrolled AUC: 0.6627
  KNN -- Graduate AUC: 0.8367
  KNN -- Macro AUC: 0.7815

  SVM -- Dropout AUC: 0.8903
  SVM -- Enrolled AUC: 0.7349
  SVM -- Graduate AUC: 0.8973
  SVM -- Macro AUC: 0.8409

  LR -- Dropout AUC: 0.9103
  LR -- Enrolled AUC: 0.7809
  LR -- Graduate AUC: 0.9188
  LR -- Macro AUC: 0.8701

  RF -- Dropout AUC: 0.9124
  RF -- Enrolled AUC: 0.8121
  RF -- Graduate AUC: 0.9280
  RF -- Macro AUC: 0.8843

  Saved:roc_curves.png


In [48]:
def match_cols(keywords, all_cols):
    matched = []
    for kw in keywords:
        for c in all_cols:
            if kw.lower() in c.lower() and c not in matched:
                matched.append(c)
    return matched

all_cols      = X_raw.columns.tolist()

academic_cols = match_cols(
    ['previous qualification (grade)', 'admission grade', 'age at enrollment',
     'daytime', 'curricular units'], all_cols)

socio_cols    = match_cols(
    ['scholarship', 'tuition fees', 'debtor', 'displaced',
     "mother's qualification", "father's qualification",
     "mother's occupation", "father's occupation"], all_cols)

macro_cols    = match_cols(
    ['gdp', 'unemployment rate', 'inflation rate'], all_cols)

demographic_cols = match_cols(
    ['marital status', 'application mode', 'application order', 'course',
     'previous qualification', 'nacionality', 'educational special needs',
     'gender', 'international'], all_cols)

# then explicitly remove any overlap with academic
demographic_cols = [c for c in demographic_cols if c not in academic_cols]
# verify
print(f"Academic      : {len(academic_cols)}  → {academic_cols}")
print(f"Socioeconomic : {len(socio_cols)}  → {socio_cols}")
print(f"Macroeconomic : {len(macro_cols)}  → {macro_cols}")
print(f"Demographic   : {len(demographic_cols)}  → {demographic_cols}")

# check nothing is double-counted or missed
all_grouped = set(academic_cols + socio_cols + macro_cols + demographic_cols)
unmatched   = [c for c in X_raw.columns if c not in all_grouped]
print(f"\nUnmatched     : {unmatched}")

Academic      : 16  → ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Daytime/evening attendance\t', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)']
Socioeconomic : 8  → ['Scholarship holder', 'Tuition fees up to date', 'Debtor', 'Displaced', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]
Macroeconomic : 3  → ['GDP', 'Unemployment rate', 'Inflation rate']
Demographic   : 9  → ['Marital status', 'Application mode', 'Application order', 'Course', 'Previous qualification', 'Nac

In [49]:

print("RQ2")

 
feature_groups = {
    'Academic'      : academic_cols,
    'Socioeconomic' : socio_cols,
    'Demographic'   : demographic_cols,
    'Macroeconomic' : macro_cols,
    'All Features'  : list(X_raw.columns),
}
 
rq2_rows = []
for grp, cols in feature_groups.items():
    if not cols:
        continue
    sc_g  = StandardScaler()
    Xg_tr = sc_g.fit_transform(X_train[cols])
    Xg_te = sc_g.transform(X_test[cols])
    row   = {'Group': grp, 'N Features': len(cols)}
    for name in MODEL_NAMES:
        m = type(best_models[name])(**best_models[name].get_params())
        met, _ = eval_model(m, Xg_tr, Xg_te, y_train, y_test)
        row[f'{name} F1'] = met['F1-Macro']
    rq2_rows.append(row)
    print(f"  {grp:15s} ({len(cols):2d} feat): "
          + "  ".join(f"{n}={row[f'{n} F1']}" for n in MODEL_NAMES))
 
df_rq2 = pd.DataFrame(rq2_rows).set_index('Group')

RQ2
  Academic        (16 feat): KNN=0.6437  SVM=0.645  LR=0.6494  RF=0.6598
  Socioeconomic   ( 8 feat): KNN=0.4678  SVM=0.4318  LR=0.4287  RF=0.4439
  Demographic     ( 9 feat): KNN=0.4248  SVM=0.3838  LR=0.3741  RF=0.4208
  Macroeconomic   ( 3 feat): KNN=0.3057  SVM=0.2221  LR=0.2221  RF=0.2221
  All Features    (36 feat): KNN=0.573  SVM=0.6433  LR=0.6882  RF=0.6959


In [50]:
print("SHAP Group Contribution  (RQ3)")


best_fitted = best_models[best_name]
X_test_df   = pd.DataFrame(X_test_sc, columns=X_raw.columns)
X_sample    = X_test_df.sample(min(200, len(X_test_df)), random_state=RANDOM_STATE)
 
if best_name == 'RF':
    explainer = shap.TreeExplainer(best_fitted)
    shap_vals = explainer.shap_values(X_sample)
    shap_abs  = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
else:
    bg        = shap.sample(X_sample, 50)
    explainer = shap.KernelExplainer(best_fitted.predict_proba, bg)
    shap_vals = explainer.shap_values(X_sample)
    shap_abs  = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
 
shap_mean = pd.Series(shap_abs.mean(axis=1), index=X_raw.columns)
 
def grp_shap(cols):
    return shap_mean[[c for c in cols if c in shap_mean.index]].sum()
 
shap_groups = {
    'Academic'      : grp_shap(academic_cols),
    'Socioeconomic' : grp_shap(socio_cols),
    'Demographic'   : grp_shap(demographic_cols),
    'Macroeconomic' : grp_shap(macro_cols),
}
print("  Summed mean |SHAP| per group:")
for g, s in sorted(shap_groups.items(), key=lambda x: -x[1]):
    print(f"    {g:15s}: {s:.4f}")

SHAP Group Contribution  (RQ3)
  Summed mean |SHAP| per group:
    Academic       : 0.3072
    Socioeconomic  : 0.0727
    Demographic    : 0.0314
    Macroeconomic  : 0.0115


In [51]:

print("VIF Screening + Reduced Feature Evaluation  (RQ4)")

 
vif_df = pd.DataFrame({
    'feature': X_raw.columns,
    'VIF'    : [variance_inflation_factor(X_train_sc, i)
                for i in range(X_train_sc.shape[1])]
}).sort_values('VIF', ascending=False)
 
print("\n  Top 10 VIF scores:")
print(vif_df.head(10).to_string(index=False))
 
high_vif     = vif_df[vif_df['VIF'] > 10]['feature'].tolist()
reduced_cols = [c for c in X_raw.columns if c not in high_vif]
X_train_red  = X_train_sc_df[reduced_cols].values
X_test_red   = X_test_sc_df[reduced_cols].values
print(f"\n  Removed (VIF > 10) : {len(high_vif)}  ->  {high_vif}")
print(f"  Remaining          : {len(reduced_cols)} features")
 
print("\n  Retraining all models on reduced feature set...")
results_red = {}
for name, model in best_models.items():
    m2 = type(model)(**model.get_params())
    met, _ = eval_model(m2, X_train_red, X_test_red, y_train, y_test)
    results_red[name] = met
 
df_red  = pd.DataFrame(results_red).T
delta   = df_red['F1-Macro'] - df_full['F1-Macro']
comp_df = pd.DataFrame({
    'Full (36)'   : df_full['F1-Macro'],
    'Reduced (30)': df_red['F1-Macro'],
    'Delta F1'    : delta
})
print("\n  RQ4 — Full vs Reduced Comparison:")
print(comp_df.to_string())

VIF Screening + Reduced Feature Evaluation  (RQ4)

  Top 10 VIF scores:
                            feature       VIF
Curricular units 1st sem (enrolled) 23.766228
Curricular units 2nd sem (enrolled) 16.985192
Curricular units 1st sem (credited) 16.787399
Curricular units 1st sem (approved) 13.350394
Curricular units 2nd sem (credited) 13.293188
Curricular units 2nd sem (approved) 10.950704
                Mother's occupation  6.216551
                Father's occupation  6.190516
   Curricular units 2nd sem (grade)  5.836489
   Curricular units 1st sem (grade)  5.336209

  Removed (VIF > 10) : 6  ->  ['Curricular units 1st sem (enrolled)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (approved)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (approved)']
  Remaining          : 30 features

  Retraining all models on reduced feature set...

  RQ4 — Full vs Reduced Comparison:
     Full (36)  Reduced (30)  Del

In [52]:
print("Generating Figures")
 
PALETTE = {'KNN': '#1e3a5f', 'SVM': '#1a7a6e', 'LR': '#b85c2a', 'RF': '#7b2d8b'}
 
# Figure 1: Results Summary (2x2)
fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle('Student Dropout Prediction — Results Summary', fontsize=14, fontweight='bold')
 
ax = axes[0, 0]
bars = ax.bar(MODEL_NAMES,
              [results_full[n]['F1-Macro'] for n in MODEL_NAMES],
              color=[PALETTE[n] for n in MODEL_NAMES])
ax.set_title('RQ1: F1-Macro — All Features (Optimised)')
ax.set_ylabel('F1-Macro'); ax.set_ylim(0, 1)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=10)
 
ax = axes[0, 1]
f1_cols = [c for c in df_rq2.columns if 'F1' in c]
sns.heatmap(df_rq2[f1_cols].astype(float), annot=True, fmt='.3f',
            cmap='YlOrRd', ax=ax, vmin=0.3, vmax=0.9, linewidths=0.5)
ax.set_title('RQ2: F1-Macro by Feature Group & Model')
ax.tick_params(axis='x', rotation=0)
 
ax = axes[1, 0]
sg = pd.Series(shap_groups).sort_values(ascending=True)
sg.plot(kind='barh', ax=ax, color=['#b85c2a', '#1a7a6e', '#1e3a5f'][:len(sg)])
ax.set_title(f'RQ3: SHAP Group Contribution ({best_name})')
ax.set_xlabel('Sum of Mean |SHAP| Values')
 
ax = axes[1, 1]
x = np.arange(len(MODEL_NAMES)); w = 0.35
ax.bar(x - w/2, df_full['F1-Macro'].values, w, label='Full (36)',    color='#1e3a5f')
ax.bar(x + w/2, df_red['F1-Macro'].values,  w, label='Reduced (30)', color='#b85c2a')
ax.set_title('RQ4: Full vs VIF-Reduced Feature Set')
ax.set_ylabel('F1-Macro'); ax.set_ylim(0, 1)
ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES); ax.legend()
 
plt.tight_layout()
fig1.savefig('fig1_results_summary.png', dpi=150, bbox_inches='tight')
plt.close(fig1)
print("  Saved: fig1_results_summary.png")
 
# Figure 4: CV scores (from GridSearchCV) + hyperparameter table
fig4, axes4 = plt.subplots(1, 2, figsize=(13, 5))
fig4.suptitle('Validation & Hyperparameter Optimisation', fontsize=13, fontweight='bold')
 
ax = axes4[0]
bars = ax.bar(MODEL_NAMES, [cv_scores[n] for n in MODEL_NAMES],
              color=[PALETTE[n] for n in MODEL_NAMES])
ax.set_title('Best CV F1-Macro from GridSearchCV (5-fold)')
ax.set_ylabel('CV F1-Macro'); ax.set_ylim(0.5, 1.0)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f'{b.get_height():.3f}', ha='center', fontsize=10)
 
ax = axes4[1]
ax.axis('off')
table = ax.table(
    cellText=[[n, str(best_params[n])] for n in MODEL_NAMES],
    colLabels=['Model', 'Best Hyperparameters'],
    cellLoc='left', loc='center'
)
table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1.2, 2)
ax.set_title('GridSearchCV — Best Parameters', pad=20)
plt.tight_layout()
fig4.savefig('fig4_validation_optimisation.png', dpi=150, bbox_inches='tight')
plt.close(fig4)
print("  Saved: fig4_validation_optimisation.png")
 
# Figure 5: SHAP summary
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, X_sample, plot_type='bar',
                  class_names=list(CLASS_NAMES), show=False, max_display=15)
plt.title(f'SHAP Feature Importance — {best_name} (Top 15)')
plt.tight_layout()
plt.savefig('fig5_shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: fig5_shap_summary.png")

Generating Figures
  Saved: fig1_results_summary.png
  Saved: fig4_validation_optimisation.png
  Saved: fig5_shap_summary.png
